# NLP Assignment 2 — Word representations for GoEmotions
**Phase 2 · Member 2 · Part B (Word2Vec, fastText) and Part C (probing and comparison)**

Member 2: *add name / ID here* · Course: *add course / section* · Date: *add date*

**How Phase 2 is split.** The Phase 2 work is divided between two team members, one notebook each. Each notebook starts from the Phase 1 output and runs **on its own**: it does not need any variable from `03a_count_based_and_packaging.ipynb`, and the two share only the code in `src/` (so they cannot disagree on the corpus, the tokens or the hyper-parameters).

| Notebook | Member | Parts | Sections |
|---|---|---|---|
| `03a_count_based_and_packaging.ipynb` | 1 | **A** and **D** | A1 – A4 Bag of Words, TF-IDF, n-grams, hand-worked example · D1 – D2 saving every model, the `vectorize()` interface |
| **`03b_embeddings_and_probing.ipynb` (this notebook)** | 2 | **B** and **C** | B1 – B4 Word2Vec (CBOW, Skip-gram), fastText · C1 – C4 neighbours, similarities, analogy, count-based vs learned |

**Topic recap.** Fine-grained emotion detection on Reddit comments (GoEmotions: 54,263 comments, 28 emotion labels). Phase 1 produced a cleaned corpus; here we turn that corpus into numbers a model can use, and compare the ways of doing it.

**Decisions that apply to everything below** (they keep the comparison fair, as the brief requires):

1. **One corpus.** Every representation is built from the Phase 1 column `text_clean`. We do *not* clean anything differently.
2. **Fitted on the training split only.** Vocabulary, IDF values and embeddings come from the 43,410 training comments. Validation and test comments are only ever *transformed*, so Phase 3 can be evaluated without leakage.
3. **Phase 1 tokens are kept as they are.** `text_clean` is already tokenised (words separated by single spaces), so every vectorizer simply splits on whitespace. A library tokenizer would silently drop one-letter words such as `i`, signal tokens such as `!` and `?`, placeholders such as `<name>` and emoji, all of which Phase 1 deliberately kept.
4. **Identical hyper-parameters** for CBOW, Skip-gram and fastText; only the training scheme (and, for fastText, the sub-word information) differs.
5. **Deterministic.** Fixed seed, one worker thread and a process-independent hash: re-running gives the same vectors.

All reusable code lives in `src/` (`representations.py`, `build_representations.py`, `vectorize.py`); this notebook calls it, so it shows exactly what the scripts do.

In [1]:
import sys, re, time, json, math, subprocess, collections, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

def find_project_root():
    """Locate the folder that contains src/cleaning.py, wherever Jupyter was started."""
    start = Path.cwd().resolve()
    for base in (start, *start.parents):
        for candidate in (base, base / "goemotions-nlp"):
            if (candidate / "src" / "cleaning.py").exists():
                return candidate
    raise FileNotFoundError(
        f"Could not find the project's src/ folder from {start}.\n"
        "This notebook needs the whole project, not just the .ipynb file: clone the repository "
        "(or unzip goemotions-nlp.zip) and open notebooks/03b_embeddings_and_probing.ipynb from inside it "
        "(on Colab/Kaggle, clone or upload the project, then %cd goemotions-nlp/notebooks first)."
    )

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

from src import representations as R

def say(text):
    """Show a sentence (with numbers computed by the code above it) as formatted text."""
    display(Markdown(text))

if not R.CLEAN_CSV.exists():
    print("Phase 1 output not found - running  python -m src.run_cleaning  ...")
    subprocess.run([sys.executable, "-m", "src.run_cleaning"], cwd=ROOT, check=True)

df = R.load_clean_corpus()
train = df[df["split"] == R.FIT_SPLIT]
texts = train["text_clean"].tolist()
sentences = R.tokenize_corpus(texts)
word_counts = collections.Counter(w for s in sentences for w in s)
N_DOCS, N_TOKENS = len(texts), sum(word_counts.values())

summary = df.groupby("split").agg(comments=("text_clean", "size"),
                                  tokens=("text_clean", lambda s: s.str.split().str.len().sum()))
summary.loc["fitted on 'train'"] = ["", ""]
display(summary.loc[["train", "validation", "test"]])
say(f"**Corpus used to fit every representation:** the *{R.FIT_SPLIT}* split — {N_DOCS:,} comments, "
    f"{N_TOKENS:,} tokens, {len(word_counts):,} distinct words, {N_TOKENS / N_DOCS:.1f} tokens per comment on average.")

,comments,tokens
split,,
train,43410,380557
validation,5426,47457
test,5427,47255


**Corpus used to fit every representation:** the *train* split — 43,410 comments, 380,557 tokens, 20,026 distinct words, 8.8 tokens per comment on average.

### A note on corpus size (it matters for Part B)

About 380 thousand training tokens is a **very small corpus** for learning word embeddings. The original Word2Vec and fastText papers train on billions of words, and the usual advice is that embeddings need at least millions. We therefore expect noisy vectors for all but the commonest words, and we interpret the neighbours and similarity scores in Parts B and C cautiously instead of claiming more than the data supports.

---
# Part B — Learned embeddings: Word2Vec (CBOW, Skip-gram) and fastText

Count-based vectors are long, sparse and treat `happy` and `glad` as unrelated columns. **Embeddings** are short, dense vectors (here 100 numbers per word) *learned* so that words used in similar contexts end up close together. Word2Vec learns them by training a tiny neural network on a prediction task over the corpus; the word vectors are the by-product.

**Hyper-parameters — identical for the three models** (the brief asks for CBOW and Skip-gram to be comparable, so only the training scheme may differ):

| Hyper-parameter | Value | Meaning |
|---|---|---|
| `vector_size` | **100** | length of each word vector |
| `window` | **5** | up to 5 words on each side of the centre word count as its context |
| `min_count` | **5** | words seen fewer than 5 times are ignored (too rare to learn from) |
| `epochs` | **30** | passes over the corpus (gensim's default is 5; we use more because the corpus is small) |
| `negative` | 5 | negative sampling: 5 random "wrong" words contrast with each true (word, context) pair |
| `sample` | 0.001 *(gensim default)* | frequent words are randomly down-sampled during training |
| `seed`, `workers` | 42, 1 | one thread and a fixed seed make training reproducible |

The values live in `src/representations.py` (`EMBED_PARAMS`), so the notebook, the build script and the saved models cannot disagree.

## B1  Word2Vec — CBOW

**CBOW** (*continuous bag of words*, `sg=0`): the network sees the context words around a position and must predict the word in the middle.

In [2]:
from gensim.models import Word2Vec, FastText

print("shared hyper-parameters:", {k: v for k, v in R.EMBED_PARAMS.items() if k != "hashfxn"})

t0 = time.time()
cbow = Word2Vec(sentences=sentences, sg=0, **R.EMBED_PARAMS)
t_cbow = time.time() - t0

say(f"**CBOW trained** in {t_cbow:.0f} s on {N_TOKENS:,} tokens. Vocabulary: **{len(cbow.wv):,} words** "
    f"(of {len(word_counts):,}; the rest occur fewer than {R.EMBED_PARAMS['min_count']} times) · "
    f"vector size {cbow.wv.vector_size} · window {cbow.window} · min_count {R.EMBED_PARAMS['min_count']} · {cbow.epochs} epochs.")

shared hyper-parameters: {'vector_size': 100, 'window': 5, 'min_count': 5, 'epochs': 30, 'negative': 5, 'workers': 1, 'seed': 42}


**CBOW trained** in 7 s on 380,557 tokens. Vocabulary: **5,205 words** (of 20,026; the rest occur fewer than 5 times) · vector size 100 · window 5 · min_count 5 · 30 epochs.

## B2  Word2Vec — Skip-gram

**Skip-gram** (`sg=1`) turns the task around: given the word in the middle, predict the words around it. Same corpus, same hyper-parameters; only `sg` changes.

In [3]:
t0 = time.time()
skipgram = Word2Vec(sentences=sentences, sg=1, **R.EMBED_PARAMS)
t_sg = time.time() - t0

for attr in ("window", "epochs", "negative"):
    assert getattr(cbow, attr) == getattr(skipgram, attr)
assert cbow.wv.vector_size == skipgram.wv.vector_size and len(cbow.wv) == len(skipgram.wv)
assert (cbow.sg, skipgram.sg) == (0, 1)

say(f"**Skip-gram trained** in {t_sg:.0f} s (CBOW took {t_cbow:.0f} s). Same vocabulary size ({len(skipgram.wv):,}), same vector size, window and epochs — "
    f"`sg` is the only difference. Skip-gram is slower because every centre word makes up to {2 * R.EMBED_PARAMS['window']} separate prediction updates, whereas CBOW makes one per position.")

**Skip-gram trained** in 21 s (CBOW took 7 s). Same vocabulary size (5,205), same vector size, window and epochs — `sg` is the only difference. Skip-gram is slower because every centre word makes up to 10 separate prediction updates, whereas CBOW makes one per position.

## B3  How the two training objectives differ

Take the cleaned comment `i love this game` with the centre word **love** and a window of 2:

* **CBOW — context → centre.** The vectors of the context words (`i`, `this`, `game`) are **averaged** into one context vector, and the network is trained to predict the single centre word `love` from it. One training example per position, and the averaging smooths out the individual context words.
* **Skip-gram — centre → context.** The network gets `love` and is trained to predict **each** context word separately: (`love` → `i`), (`love` → `this`), (`love` → `game`). Several training examples per position, and no averaging.

**Which should do better on our corpus?** *(our expectation, written before looking at any results)* The usual rule of thumb is that **Skip-gram does better for small corpora and rare words**, because each occurrence of a word creates several updates, so a rare word gets more training signal; CBOW is faster and tends to be slightly better for very frequent words, because averaging the context reduces noise. Our corpus is small (about 380 thousand tokens) and its vocabulary is long-tailed, so we **expect Skip-gram to win**. We test that expectation in section C2 instead of assuming it.

## B4  fastText and sub-word information

Word2Vec gives every word **one atomic vector**: `love` and `loved` are two unrelated rows, and a word that is not in the vocabulary has no vector at all. **fastText** represents a word as the **sum of the vectors of its character n-grams** (here every 3- to 6-character piece of `<word>`, with `<` and `>` marking the word boundaries) plus the word itself. Consequences:

* words that share pieces share parameters (`love`, `loved`, `lovely` all contain `<lov` and `ove`), so rare words borrow strength from related words;
* a word **never seen in training** (a rare word, a misspelling) can still be given a vector, built from the n-grams it shares with known words;
* the price: words that merely *look* alike can come out close (`love` / `glove`), and the n-gram table needs memory.

We train fastText with the **Skip-gram scheme and the same hyper-parameters as the Word2Vec models**, so the only difference from the Skip-gram model is the sub-word information. (`bucket=50,000` is the number of hash slots for the n-grams; gensim's default of 2 million would take about 800 MB for this small corpus.)

In [4]:
t0 = time.time()
fasttext = FastText(sentences=sentences, sg=1, **R.EMBED_PARAMS, **R.FASTTEXT_EXTRA)
t_ft = time.time() - t0

def char_ngrams(word, n_min=3, n_max=6):
    w = f"<{word}>"
    return [w[i:i + n] for n in range(n_min, n_max + 1) for i in range(len(w) - n + 1)]

all_grams = {g for w in fasttext.wv.key_to_index for g in char_ngrams(w)}
say(f"**fastText trained** in {t_ft:.0f} s. Word vocabulary: {len(fasttext.wv):,} words (same as Word2Vec) plus a table of "
    f"{fasttext.wv.vectors_ngrams.shape[0]:,} n-gram vectors; the vocabulary's words contain {len(all_grams):,} distinct character n-grams, "
    f"so some n-grams share a hash slot (a small loss in precision that keeps the saved model at about 22 MB).")

for w in ["terrible", "terible"]:
    print(f"{w:9s} -> {len(char_ngrams(w)):2d} n-grams, e.g. {char_ngrams(w)[:8]}")
for typo, correct in [("terible", "terrible"), ("luv", "love")]:
    a, b = set(char_ngrams(correct)), set(char_ngrams(typo))
    print(f"\n'{typo}' shares {len(a & b)} of its {len(b)} character n-grams with '{correct}' -> {len(a & b) / len(b):.0%}")

**fastText trained** in 35 s. Word vocabulary: 5,205 words (same as Word2Vec) plus a table of 50,000 n-gram vectors; the vocabulary's words contain 39,250 distinct character n-grams, so some n-grams share a hash slot (a small loss in precision that keeps the saved model at about 22 MB).

terrible  -> 26 n-grams, e.g. ['<te', 'ter', 'err', 'rri', 'rib', 'ibl', 'ble', 'le>']
terible   -> 22 n-grams, e.g. ['<te', 'ter', 'eri', 'rib', 'ibl', 'ble', 'le>', '<ter']

'terible' shares 13 of its 22 character n-grams with 'terrible' -> 59%

'luv' shares 0 of its 6 character n-grams with 'love' -> 0%


In [5]:
misspelled = ["terible", "awsome", "hillarious", "depresed", "gratefull", "excitment"]
rare_in_corpus = ["definately", "luv", "yayy", "cutee"]
control = ["thx"]

def w2v_neighbours(model, word, n=5):
    try:
        return ", ".join(w for w, _ in model.wv.most_similar(word, topn=n))
    except KeyError:
        return "KeyError - word not in vocabulary"

rows = []
for kind, words in [("misspelled (not in corpus)", misspelled), ("rare (in corpus, below min_count)", rare_in_corpus), ("control (in vocabulary)", control)]:
    for w in words:
        rows.append({"query word": w, "kind": kind, "count in corpus": word_counts.get(w, 0),
                     "Word2Vec Skip-gram": w2v_neighbours(skipgram, w), "fastText": w2v_neighbours(fasttext, w)})
oov_table = pd.DataFrame(rows).set_index("query word")
pd.set_option("display.max_colwidth", 70)
display(oov_table)

v_typo, v_good = fasttext.wv["terible"], fasttext.wv["terrible"]
cos = float(np.dot(v_typo, v_good) / (np.linalg.norm(v_typo) * np.linalg.norm(v_good)))
say(f"The fastText vector of the misspelling `terible` has cosine similarity **{cos:.2f}** with the vector of `terrible`, "
    f"although `terible` never occurs in the corpus. Word2Vec cannot say anything about it.")

,kind,count in corpus,Word2Vec Skip-gram,fastText
query word,,,,
terible,misspelled (not in corpus),0,KeyError - word not in vocabulary,"terrible, horrible, feasible, audible, bible"
awsome,misspelled (not in corpus),0,KeyError - word not in vocabulary,"awesome, handsome, likeable, epitome, wholesome"
hillarious,misspelled (not in corpus),0,KeyError - word not in vocabulary,"hilarious, various, hilariously, mysterious, furious"
depresed,misspelled (not in corpus),0,KeyError - word not in vocabulary,"depressed, depress, depressing, depression, used"
gratefull,misspelled (not in corpus),0,KeyError - word not in vocabulary,"grateful, hateful, circlejerk, thankyou, integrate"
excitment,misspelled (not in corpus),0,KeyError - word not in vocabulary,"excitement, commitment, appointment, excite, excited"
definately,"rare (in corpus, below min_count)",3,KeyError - word not in vocabulary,"definitely, deliberately, ultimately, lately, legitimately"
luv,"rare (in corpus, below min_count)",2,KeyError - word not in vocabulary,"life, luck, lift, luxury, cali"
yayy,"rare (in corpus, below min_count)",3,KeyError - word not in vocabulary,"yay, ayy, aye, yawn, !"


The fastText vector of the misspelling `terible` has cosine similarity **0.86** with the vector of `terrible`, although `terible` never occurs in the corpus. Word2Vec cannot say anything about it.

**What the demonstration shows.**

* **Word2Vec fails on every word outside its vocabulary.** It raises a `KeyError`: no vector exists, so a Phase 3 pipeline must either skip such words or crash. Note that this also hits *rare words that do occur in our corpus* (`definately`, `luv`, ...), because `min_count=5` discarded them.
* **fastText answers, usually sensibly, but not always.** For the misspellings it returns the intended word at rank 1 (`terible` → `terrible`, `awsome` → `awesome`, `hillarious` → `hilarious`, `depresed` → `depressed`, `gratefull` → `grateful`, `excitment` → `excitement`), and it recovers the rare in-corpus spellings `definately` → `definitely`, `yayy` → `yay` and `cutee` → `cute`, because they share most of their character n-grams with the correct word. **It fails for `luv`:** that slang spelling shares *no* character n-gram with `love` (see the overlap printed above), so the neighbours are look-alikes (`life`, `luck`, `lift`). Sub-words only help when the spelling stays close to the correct word.
* **It also helps for words that *are* in the vocabulary but rare.** For the control word `thx` (17 occurrences) fastText returns `thankyou`, `thanks`, `thank`, whereas Skip-gram returns unrelated words (`gotcha`, `clarification`): too few examples to learn from, but fastText shares evidence with `thanks`-like spellings.
* **The neighbours beyond the first are often just look-alikes** (`feasible`, `audible`, `bible` for `terible`; `commitment`, `appointment` for `excitment`). Sub-word similarity is *orthographic*, not semantic, so fastText's lower-ranked neighbours should be trusted less. Section C1 shows the same effect for in-vocabulary words (`love` → `glove`).

---
# Part C — Probing and comparison

How do we know what an embedding "knows"? We look at its nearest neighbours (words with the highest **cosine similarity** to the query word), at similarity scores for hand-picked word pairs, and at one analogy. Because the corpus is small, treat all of this as exploratory.

## C1  Nearest neighbours

Five query words that matter for emotion detection, shown for **CBOW, Skip-gram and fastText** (top 8 neighbours each).

In [6]:
models = {"CBOW": cbow.wv, "Skip-gram": skipgram.wv, "fastText": fasttext.wv}
QUERIES = ["love", "sad", "angry", "thank", "funny"]

def neighbours(kv, word, n=8):
    return [w for w, _ in kv.most_similar(word, topn=n)]

for q in QUERIES:
    say(f"**`{q}`**  (occurs {word_counts[q]:,} times in the training corpus)")
    display(pd.DataFrame({name: neighbours(kv, q) for name, kv in models.items()}, index=range(1, 9)))

**`love`**  (occurs 2,222 times in the training corpus)

,CBOW,Skip-gram,fastText
1,hate,freakin,loved
2,remind,😍,lover
3,roast,yess,glove
4,awesome,awesome,adore
5,amazing,mini,😍
6,hat,melt,metallica
7,beautiful,kitten,sunny
8,recommend,commentary,adorable


**`sad`**  (occurs 293 times in the training corpus)

,CBOW,Skip-gram,fastText
1,upset,bitter,<sad>
2,disappointing,relatable,mad
3,sensitive,naive,nervous
4,uncomfortable,delusional,mood
5,angry,overweight,kudo
6,insecure,immature,petty
7,strange,nauseous,disappointing
8,cringe,idk,disappointed


**`angry`**  (occurs 84 times in the training corpus)

,CBOW,Skip-gram,fastText
1,upset,cane,hungry
2,uncomfortable,unattractive,ant
3,awkward,cringe,miserable
4,cringe,sensitive,cringy
5,toxic,sociopath,motivated
6,rich,elderly,disturb
7,attractive,doxxe,loud
8,annoying,cringy,cringe


**`thank`**  (occurs 2,244 times in the training corpus)

,CBOW,Skip-gram,fastText
1,appreciate,thankyou,thankyou
2,thanks,☺,thanks
3,apology,appreciate,thankful
4,helpful,thorough,thx
5,glad,😀,helpful
6,cheer,encouragement,gotcha
7,☺,😆,thankfully
8,prayer,sincerely,appreciate


**`funny`**  (occurs 295 times in the training corpus)

,CBOW,Skip-gram,fastText
1,awkward,joke,bunny
2,obvious,pronounce,sunny
3,attractive,cringy,funniest
4,annoying,dense,intrigue
5,cringy,coincidence,joke
6,intrigue,liner,💖
7,creepy,intrigue,fair
8,sad,oop,asexual


**Do the neighbours make sense?**

* **`thank`** — the clearest success: all three models return words of gratitude (`thanks`, `thankyou`, `appreciate`, `thankful`). It is frequent and almost always used in the same way, so its contexts are consistent.
* **`sad`** — CBOW gives a coherent cluster of negative feeling (`upset`, `disappointing`, `uncomfortable`, `angry`, `insecure`). Skip-gram's list is much noisier (`bitter`, `relatable`, `naive`), and fastText's first neighbour is `<sad>`, the placeholder Phase 1 uses for the `:(` emoticon, which is a genuinely sensible neighbour.
* **`love`** — a mixed picture. Genuinely positive neighbours appear (`awesome`, `amazing`, `adorable`, the heart-eyes emoji), mixed with noise (`hat`, `roast`, `mini`, `metallica`). CBOW even ranks its antonym `hate` first (see C2). fastText ranks `loved` and `lover` high (shared sub-words) but also **`glove`**, a pure spelling coincidence.
* **`angry`** and **`funny`** — weaker. `angry` is rarer (84 occurrences) and `funny` is probably used in several senses (humorous, odd, sarcastic: CBOW's neighbours `awkward`, `cringy`, `creepy` suggest the "odd" sense); the neighbours are loosely related at best, and fastText returns look-alikes (`hungry` for `angry`; `bunny`, `sunny` for `funny`).
* **Overall:** neighbours are trustworthy for frequent words with a stable usage and unreliable for rarer or ambiguous ones, which is what a ~380 k-token corpus predicts. In the run saved in this notebook, CBOW's neighbours are the most coherent for `sad` and `angry`, and Skip-gram's are the noisiest.

## C2  Similar and unrelated word pairs

Two pairs we expect to be **similar** (`happy`–`glad`, `funny`–`hilarious`) and two we expect to be **unrelated** (`table`–`happy`, `pizza`–`sad`), with the cosine similarity under each model. Two **antonym** pairs follow, because they test something that matters for emotions.

In [7]:
def cos_table(pairs):
    return pd.DataFrame([{"pair": f"{a} – {b}", **{name: round(float(kv.similarity(a, b)), 3) for name, kv in models.items()}}
                         for a, b in pairs]).set_index("pair")

similar   = [("happy", "glad"), ("funny", "hilarious")]
unrelated = [("table", "happy"), ("pizza", "sad")]
antonyms  = [("love", "hate"), ("good", "bad")]

print("expected SIMILAR");    display(cos_table(similar))
print("expected UNRELATED");  display(cos_table(unrelated))
print("ANTONYMS");            display(cos_table(antonyms))

expected SIMILAR


,CBOW,Skip-gram,fastText
pair,,,
happy – glad,0.439,0.322,0.310
funny – hilarious,0.406,0.388,0.449


expected UNRELATED


,CBOW,Skip-gram,fastText
pair,,,
table – happy,-0.130,0.118,0.189
pizza – sad,-0.224,0.106,0.107


ANTONYMS


,CBOW,Skip-gram,fastText
pair,,,
love – hate,0.481,0.478,0.422
good – bad,0.428,0.437,0.430


**Discussion.**

* **Similar pairs score higher than unrelated pairs under every model**, as we hoped. But look at the *scale*: the absolute cosines differ a lot between models (Skip-gram and fastText give the unrelated pairs positive scores of about 0.1–0.2, CBOW gives values below zero), so a raw cosine should **not** be compared across models. What carries information is the *gap* between related and unrelated pairs within one model.
* **Antonyms are close.** `love`–`hate` and `good`–`bad` have similarities comparable to genuine synonyms. This is not a bug: embeddings capture *distributional* similarity, and `love` and `hate` occur in the same contexts ("I ___ this", "I ___ it when ..."). For **emotion** detection this is an important warning: an embedding places opposite emotions near each other, so the positive/negative *polarity* is not what the geometry encodes best.

A hand-picked pair of pairs is anecdotal, so the next cell repeats the test with **8 related and 8 unrelated pairs** (all words occur at least 5 times) and measures the gap.

In [8]:
related_pairs = [("happy", "glad"), ("funny", "hilarious"), ("angry", "mad"), ("scared", "afraid"),
                 ("awful", "terrible"), ("disgusting", "gross"), ("sad", "upset"), ("cute", "sweet")]
unrelated_pairs = [("table", "happy"), ("pizza", "sad"), ("rain", "funny"), ("car", "thank"),
                   ("hockey", "angry"), ("table", "afraid"), ("pizza", "awful"), ("rain", "glad")]
assert all(word_counts[w] >= 5 for p in related_pairs + unrelated_pairs for w in p)

def benchmark(kv):
    rel = np.array([kv.similarity(a, b) for a, b in related_pairs])
    unr = np.array([kv.similarity(a, b) for a, b in unrelated_pairs])
    return {"related (mean cos)": rel.mean(), "unrelated (mean cos)": unr.mean(),
            "gap": rel.mean() - unr.mean(), "pairs ranked correctly (of 8)": int((rel > unr).sum())}

bench = pd.DataFrame({name: benchmark(kv) for name, kv in models.items()}).T.round(3)
bench["pairs ranked correctly (of 8)"] = bench["pairs ranked correctly (of 8)"].astype(int)
display(bench)
best = bench["gap"].idxmax()
say(f"**Result.** The model that separates related from unrelated pairs best is **{best}** (gap {bench.loc[best, 'gap']:.3f}); "
    f"the others have gaps of {', '.join(f'{v:.3f} ({k})' for k, v in bench['gap'].items() if k != best)}. "
    f"All three rank most pairs in the right order. Our B3 expectation was that Skip-gram would win; "
    f"{'it did.' if best == 'Skip-gram' else 'it did **not**. The next cell checks whether that depends on the number of epochs.'}")

,related (mean cos),unrelated (mean cos),gap,pairs ranked correctly (of 8)
CBOW,0.456,-0.126,0.581,8
Skip-gram,0.355,0.142,0.214,7
fastText,0.378,0.175,0.203,8


**Result.** The model that separates related from unrelated pairs best is **CBOW** (gap 0.581); the others have gaps of 0.214 (Skip-gram), 0.203 (fastText). All three rank most pairs in the right order. Our B3 expectation was that Skip-gram would win; it did **not**. The next cell checks whether that depends on the number of epochs.

In [9]:
rows = []
for ep in (5, 10, 30, 60):
    for sg_flag, label in ((0, "CBOW"), (1, "Skip-gram")):
        if ep == 30:
            kv = (cbow if sg_flag == 0 else skipgram).wv
        else:
            kv = Word2Vec(sentences=sentences, sg=sg_flag, **{**R.EMBED_PARAMS, "epochs": ep}).wv
        rows.append({"epochs": ep, "model": label, "gap": round(benchmark(kv)["gap"], 3)})
sens = pd.DataFrame(rows).pivot(index="epochs", columns="model", values="gap")
display(sens)

model,CBOW,Skip-gram
epochs,,
5,0.161,0.211
10,0.420,0.319
30,0.581,0.214
60,0.486,0.199


**What this tells us about B3.** *(read with the table above; the numbers come from the code, the reasoning is ours)*

* The textbook expectation holds **only when training is short**: with very few epochs Skip-gram, which makes more updates per word, is ahead of CBOW. With enough epochs CBOW overtakes it, and in our run it stays ahead.
* A plausible explanation (we did **not** test it) is that CBOW's averaging of the context words smooths the noise in a small, noisy corpus of short Reddit comments, while Skip-gram, which fits every (word, context) pair separately, is more prone to fitting noise once it has had many epochs.
* **Therefore:** on *our* data and with the identical hyper-parameters the brief requires, our expectation was only partly right. We should not claim that one scheme is better in general: the ranking depends on training length, on a small hand-made benchmark, and on one corpus. For Phase 3 both are saved, and the downstream task can decide.

## C3  An analogy

The classic test is *a − b + c ≈ ?* (`king − man + woman ≈ queen`). Our domain is emotion, so the main attempt is **love − hate + sad ≈ happy** ("`sad` is to `love` as `hate`…" is awkward, but it asks whether the *valence direction* from `hate` to `love` carries over to `sad`). We also try a family analogy and a degree analogy. For each we report the top-5 answers and the **rank of the expected word** among the whole vocabulary (1 = first). Failures are reported as they are.

In [10]:
def analogy(kv, a, b, c, expected, topn_show=5):
    """a - b + c: vector arithmetic, answers ranked by cosine similarity; the three query words are excluded by gensim."""
    ranked = [w for w, _ in kv.most_similar(positive=[a, c], negative=[b], topn=len(kv))]
    return ranked[:topn_show], ranked.index(expected) + 1

tests = [("love", "hate", "sad", "happy"), ("man", "woman", "mom", "dad"), ("good", "bad", "terrible", "great")]
rows = []
for a, b, c, expected in tests:
    for name, kv in models.items():
        top5, rank = analogy(kv, a, b, c, expected)
        rows.append({"analogy": f"{a} − {b} + {c} ≈ {expected}?", "model": name, "top 5 answers": ", ".join(top5),
                     f"rank of expected word (of {len(kv):,})": rank})
analogies = pd.DataFrame(rows).set_index(["analogy", "model"])
pd.set_option("display.max_colwidth", 80)
display(analogies)

top 5 answers  rank of expected word (of 5,205)
analogy                        model                                                                                      
love − hate + sad ≈ happy?     CBOW              beautiful, happy, funny, excited, upset                                 2
                               Skip-gram  story, bitter, rose, admittedly, embarrassment                               187
                               fastText       story, malicious, skyrim, metallica, loved                              1467
man − woman + mom ≈ dad?       CBOW             dad, goodness, buddy, brother, boyfriend                                 1
                               Skip-gram                dad, condolence, mayo, nephew, 💖                                 1
                               fastText        momma, dad, grandparent, grandpa, grandma                                 2
good − bad + terrible ≈ great? CBOW               amazing, messy, dna, fantastic, former                                17
                               Skip-gram        horrible, segment, shout, luxury, ending                               881
                               fastText     tribute, horrible, bible, possible, feasible                               719

**Result, honestly.**

* **The emotion analogy mostly fails.** `love − hate + sad ≈ happy` finds `happy` near the top only for CBOW; Skip-gram and fastText put it far down the ranking, and the top answers are unrelated words. The degree analogy (`good − bad + terrible ≈ great`) also fails: the expected word is not among the top 10 answers of any model.
* **The family analogy works** (`man − woman + mom ≈ dad` returns `dad` first or second), because family words are frequent and their contexts are regular.
* **Why it fails for emotions:** analogies need clean, linear relations between words, and that requires large corpora. With 380 k tokens, short comments, and antonyms that share contexts (C2), the `love → hate` direction is not a consistent "valence" axis. One success out of three attempts must not be over-interpreted either: the family success shows the method can work on regular, frequent words, not that these embeddings "understand" relations.

## C4  Count-based vs learned representations on our data

The numbers in the table are measured on this corpus (vectors of the saved TF-IDF settings: unigrams + bigrams, `min_df=3`). The out-of-vocabulary rate is measured on the **validation** split, i.e. on text the models have not seen.

In [11]:
# Bag of Words is built in notebook 03a (A1); it is rebuilt here (about a second) so this notebook runs on its own.
bow = R.make_count_vectorizer()
X_bow = bow.fit_transform(texts)

tfidf_final = R.slim(R.make_tfidf_vectorizer(**R.TFIDF_FINAL).fit(texts))
X_final = tfidf_final.transform(texts)

val_sentences = R.tokenize_corpus(df.loc[df["split"] == "validation", "text_clean"])
val_tokens = [w for s in val_sentences for w in s]
oov_bow = np.mean([w not in bow.vocabulary_ for w in val_tokens])
oov_w2v = np.mean([w not in cbow.wv.key_to_index for w in val_tokens])
docs_with_oov_bow = np.mean([any(w not in bow.vocabulary_ for w in s) for s in val_sentences])
docs_with_oov_w2v = np.mean([any(w not in cbow.wv.key_to_index for w in s) for s in val_sentences])
docs_all_oov_w2v = np.mean([all(w not in cbow.wv.key_to_index for w in s) for s in val_sentences])

def sparse_bytes_per_doc(X): return (X.data.nbytes + X.indices.nbytes) / X.shape[0]

def zero_pct(kv):
    return 100 * np.mean(np.vstack([R.document_vector(s, kv) for s in sentences[:2000]]) == 0)
comparison = pd.DataFrame({
    "dimensions": [f"{X_bow.shape[1]:,}", f"{X_final.shape[1]:,}", "100", "100", "100"],
    "zeros in a document vector": [f"{R.sparsity_pct(X_bow):.2f} %", f"{R.sparsity_pct(X_final):.2f} %",
                                   f"{zero_pct(cbow.wv):.1f} %", f"{zero_pct(skipgram.wv):.1f} %", f"{zero_pct(fasttext.wv):.1f} %"],
    "memory per document": [f"{sparse_bytes_per_doc(X_bow):.0f} B", f"{sparse_bytes_per_doc(X_final):.0f} B", "400 B", "400 B", "400 B"],
    "unseen word": ["ignored (no column)", "ignored (no column)", "no vector: KeyError / skipped", "no vector: KeyError / skipped", "vector built from character n-grams"],
}, index=["Bag of Words (unigrams)", "TF-IDF (uni+bigrams, min_df=3)", "Word2Vec CBOW", "Word2Vec Skip-gram", "fastText"])
display(comparison)

say(f"**Out-of-vocabulary on the validation split.** {100 * oov_bow:.1f} % of validation tokens are not in the Bag-of-Words vocabulary "
    f"({100 * docs_with_oov_bow:.0f} % of comments contain at least one such word); for Word2Vec (`min_count=5`) it is {100 * oov_w2v:.1f} % of tokens "
    f"and {100 * docs_with_oov_w2v:.0f} % of comments, and {100 * docs_all_oov_w2v:.1f} % of comments have *no* word at all in the Word2Vec vocabulary (their mean vector is all zeros). "
    f"fastText returns a vector for 100 % of tokens, although the vectors of unseen words are only as good as the n-grams they share with known words.")

,dimensions,zeros in a document vector,memory per document,unseen word
Bag of Words (unigrams),"20,026",99.96 %,66 B,ignored (no column)
"TF-IDF (uni+bigrams, min_df=3)","22,910",99.95 %,138 B,ignored (no column)
Word2Vec CBOW,100,0.1 %,400 B,no vector: KeyError / skipped
Word2Vec Skip-gram,100,0.1 %,400 B,no vector: KeyError / skipped
fastText,100,0.0 %,400 B,vector built from character n-grams


**Out-of-vocabulary on the validation split.** 2.7 % of validation tokens are not in the Bag-of-Words vocabulary (19 % of comments contain at least one such word); for Word2Vec (`min_count=5`) it is 7.0 % of tokens and 40 % of comments, and 0.1 % of comments have *no* word at all in the Word2Vec vocabulary (their mean vector is all zeros). fastText returns a vector for 100 % of tokens, although the vectors of unseen words are only as good as the n-grams they share with known words.

**Comparison.**

| | Count-based (BoW, TF-IDF, n-grams) | Learned (Word2Vec, fastText) |
|---|---|---|
| **Dimensionality** | one per vocabulary entry: tens of thousands, growing with n-grams | fixed and small (100), chosen by us |
| **Sparsity** | > 99.9 % zeros, so it must be stored sparse | dense: every number is used |
| **Unseen words** | simply ignored (no column), which silently loses information | Word2Vec: no vector; fastText: built from sub-words |
| **What it captures** | *which* words occur and how distinctive they are (TF-IDF), plus a little local order (n-grams). Two comments are similar only if they share **exactly the same words**: `happy` and `glad` are unrelated columns | *meaning by usage*: words in similar contexts get similar vectors, so `happy` and `glad` are close even if never seen together. But only if there is enough data, and it also puts antonyms (`love`/`hate`) close |
| **Document vector** | the representation of the document itself | needs a pooling step: we average the word vectors, which discards word order |
| **Interpretability** | each dimension is a word or n-gram: easy to inspect | dimensions have no individual meaning |

**What this means for our data.** The comments are short (about 9 tokens after cleaning), so each count-based vector has only about 8–12 non-zero entries and little overlap with other comments, while the learned vectors can link different words with similar meaning. On the other hand, the embeddings are trained on a very small corpus (see the note above), so their quality is limited, and (as measured above) a few percent of validation tokens still have no Word2Vec vector. Which representation is better for emotion classification is a question for Phase 3; the table shows what each can and cannot do before any classifier is involved.

### Check against the saved models (link to D1)

Part D (notebook `03a`, Member 1) saves the embeddings with `python -m src.build_representations`. Training is deterministic, so the saved files must hold exactly the vectors trained above. If `models/` is empty, the cell first builds the three embeddings with the script.

In [12]:
embed_names = ("cbow", "skipgram", "fasttext")
if not all((R.MODELS_DIR / R.MODEL_FILES[n]).exists() for n in embed_names):
    subprocess.run([sys.executable, "-m", "src.build_representations", "--only", *embed_names], cwd=ROOT, check=True)

saved = {name: R.load_vectors(R.MODELS_DIR / R.MODEL_FILES[name]) for name in embed_names}
trained = {"cbow": cbow.wv, "skipgram": skipgram.wv, "fasttext": fasttext.wv}
for name in saved:
    same = np.array_equal(saved[name].vectors, trained[name].vectors) and saved[name].index_to_key == trained[name].index_to_key
    print(f"{name:9s} saved vectors identical to the notebook's model: {same}")
    assert same
assert np.array_equal(saved["fasttext"].vectors_ngrams, fasttext.wv.vectors_ngrams)
print("fastText n-gram table identical too")

cbow      saved vectors identical to the notebook's model: True
skipgram  saved vectors identical to the notebook's model: True
fasttext  saved vectors identical to the notebook's model: True
fastText n-gram table identical too


### Summary (Member 2)

* **Part B.** CBOW, Skip-gram and fastText were trained with identical hyper-parameters on a small (about 380 k-token) corpus; fastText handles misspelled and unseen words, Word2Vec does not.
* **Part C.** Neighbours and similarities are sensible for frequent, consistent words and unreliable otherwise; antonyms are close; analogies mostly fail on this corpus; which of CBOW and Skip-gram is better depends on training length.

Parts A and D (Bag of Words, TF-IDF, n-grams, the hand-worked example, the saved models and `vectorize()`) are in `03a_count_based_and_packaging.ipynb`.